# Modelo Dinámico Simple de Pilas SAG — Balance de Masa
## División El Teniente — Codelco
**Período de análisis:** Enero–Junio 2026

---

### Ecuaciones del modelo

**Escenario A — Operación normal (sin ventana T8):**
$$\frac{dS_i}{dt} = Q_{\text{in},i}(t) - Q_{\text{out},i}(t)$$

**Escenario B — Ventana T8 activa (Qin = 0):**
$$\frac{dS_i}{dt} = -\text{rate}_i \quad [\%/h]$$
$$\text{rate}_i = \frac{Q_{\text{SAG},i}}{\text{Cap}_i} \times 100$$

---

### Parámetros calibrados (Fase 8 ODE)

| | SAG1 | SAG2 |
|--|--|--|
| Capacidad pila | 38,685 ton | 98,401 ton |
| Tasa P50 | 2.775 %/h | 2.270 %/h |
| Autonomía P50 desde 100% | ~36 h | ~44 h |

In [ ]:
import subprocess, sys
from pathlib import Path

BASE = Path('C:/Users/jorel038/OneDrive - Codelco/Documentos/AA_CIO_DET/07_Rendimientos')

print('Ejecutando modelo dinamico de pilas...')
r = subprocess.run(
    [sys.executable, '-X', 'utf8', str(BASE / 'src/modelo_dinamico_pilas.py')],
    capture_output=True, text=True, encoding='utf-8'
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

FIG_DIR = BASE / 'outputs/figures/modelo_dinamico_pilas'

figuras = [
    ('01_balance_pila_sag1_sin_ventana.png',  'Fig 01 — Balance histórico SAG1'),
    ('02_balance_pila_sag2_sin_ventana.png',  'Fig 02 — Balance histórico SAG2'),
    ('03_deplecion_pila_sag1_ventana_t8.png', 'Fig 03 — Depleción SAG1 con ventana T8'),
    ('04_deplecion_pila_sag2_ventana_t8.png', 'Fig 04 — Depleción SAG2 con ventana T8'),
    ('05_autonomia_por_nivel_inicial_sag1.png','Fig 05 — Autonomía SAG1'),
    ('06_autonomia_por_nivel_inicial_sag2.png','Fig 06 — Autonomía SAG2'),
    ('07_matriz_riesgo_sag1.png',              'Fig 07 — Matriz de riesgo SAG1'),
    ('08_matriz_riesgo_sag2.png',              'Fig 08 — Matriz de riesgo SAG2'),
    ('09_umbral_critico_pila_sag1.png',        'Fig 09 — Umbral crítico SAG1'),
    ('10_umbral_critico_pila_sag2.png',        'Fig 10 — Umbral crítico SAG2'),
]

for fname, title in figuras:
    fpath = FIG_DIR / fname
    if not fpath.exists():
        print(f'No encontrado: {fname}')
        continue
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(mpimg.imread(str(fpath)))
    ax.axis('off')
    ax.set_title(title, fontsize=11, pad=8)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

XLS = BASE / 'outputs/excel/modelo_dinamico_pilas.xlsx'

print('=== TASAS DE CONSUMO ===')
df_tasas = pd.read_excel(XLS, sheet_name='Tasas_Consumo')
print(df_tasas.to_string(index=False))

print('\n=== INVENTARIO MINIMO PRE-VENTANA (para terminar en >= 20%) ===')
df_inv = pd.read_excel(XLS, sheet_name='Inventario_Minimo')
pivot = df_inv[df_inv.percentil=='P50'].pivot_table(
    index='SAG', columns='duracion_ventana_h',
    values='nivel_min_para_terminar_en_20pct'
)
print(pivot.round(1).to_string())

print('\n=== METRICAS P50 — ventana 8h ===')
df_m = pd.read_excel(XLS, sheet_name='Resumen_P50')
df_8h = df_m[df_m.duracion_ventana==8].sort_values(['SAG','nivel_inicial_pila'])
cols = ['SAG','nivel_inicial_pila','nivel_final_pila','caida_pila_pct',
        'tiempo_hasta_40','tiempo_hasta_30','tiempo_hasta_20','autonomia_horas']
print(df_8h[cols].to_string(index=False))

In [ ]:
# Mostrar reporte Markdown
RPT = BASE / 'outputs/reports/resumen_modelo_dinamico_pilas.md'
from IPython.display import Markdown, display
display(Markdown(RPT.read_text(encoding='utf-8')))